### Assemble the data-quality input table
- Silver's orders table has no amount column (amount lives in items), so we add
  `order_amount` = each order's total item price.
- Inner join keeps only orders that actually have items (a real amount to check).
- Exports one clean CSV so Great Expectations can validate it locally, where the
  HTML report opens straight in a browser.
- **Data quality gate** = an automated check that must pass before we trust Silver
  enough to build Gold on top of it.

In [0]:
from pyspark.sql.functions import sum as _sum, round as _round

orders_clean = spark.table("workspace.silver.orders_clean")
items        = spark.table("workspace.silver.order_items_enriched")

order_amount = (items.groupBy("order_id")
                .agg(_round(_sum("price"), 2).alias("order_amount")))   # per-order revenue

orders_dq = (orders_clean
    .join(order_amount, on="order_id", how="inner")                     # orders that have an amount
    .select("order_id", "customer_id", "customer_unique_id",
            "order_purchase_date", "order_status", "order_amount"))

orders_dq.toPandas().to_csv("/Volumes/workspace/bronze/raw/orders_dq.csv", index=False)
print("Exported", orders_dq.count(), "rows to orders_dq.csv")

Exported 98666 rows to orders_dq.csv
